In [ ]:
!pip install pandas
%pip install deep-translator


In [ ]:
%pip install matplotlib
%pip install seaborn
%pip install scikit-learn

In [ ]:
!python.exe -m pip install --upgrade pip

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from deep_translator import GoogleTranslator

print("Hello world")

In [ ]:
df=pd.read_csv("D:\data e\Ecomerce-data-pipeline\Medallion Architecture data\Bronze\olist_order_reviews_dataset.csv")
df.head()

In [ ]:
print(df.columns)
print(df.isnull().sum())

In [ ]:
print(df['review_comment_message'].head())

In [ ]:
review=df['review_comment_message'].dropna()
review.head()

In [ ]:
review=review[review.str.len()>20]
print(review.count())
review=review.sample(100)
review.head()


In [ ]:
translated_reviews = review.apply(lambda x: GoogleTranslator(source='auto', target='en').translate(x))
translated_reviews.head()

In [ ]:
%pip install nltk
import re
import nltk
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)

    #remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove stop words
    text = ' '.join(word for word in text.split() if word not in stop_words)
    
    return text

In [ ]:
review_cleaned = translated_reviews.apply(preprocess_text)
review_cleaned.head()

In [ ]:
for i in range(5):
    print("Original:", review.iloc[i])
    print("Translated:", translated_reviews.iloc[i])
    print("Cleaned :", review_cleaned.iloc[i])
    print("-" * 50)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
v=TfidfVectorizer()
transformed_output=v.fit_transform(review_cleaned)
print(v.vocabulary_)
all_features = v.get_feature_names_out()
transformed_output.toarray()[:2]

In [ ]:
for word in all_features:  # Print the first 10 features
    index = v.vocabulary_.get(word)
    print(f" {word}: {v.idf_[index]}")

    

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(transformed_output)
print(similarity_matrix[:30, :5])

In [ ]:
results = []

threshold = 0.7

for i in range(len(similarity_matrix)):
    for j in range(i+1, len(similarity_matrix)):
        
        score = similarity_matrix[i][j]
        
        if score > threshold:
            
            results.append({
                "Review_1": review_cleaned.iloc[i],
                "Review_2": review_cleaned.iloc[j],
                "Similarity_Score": round(score, 2)
            })

In [ ]:
results_df = pd.DataFrame(results)

print(results_df.head())
results_df.to_csv("similar_reviews.csv", index=False)